In [1]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langgraph.graph import StateGraph,START, END,MessagesState
from typing import TypedDict,Annotated,Literal
import os 
import sys
sys.path.insert(1, r'D:\Notebooks\LLM\env')
from enviorment import load_env
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
#from pydirectoryloader import rag_function
import os 
load_env()
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import GoogleSerperAPIWrapper,WikipediaAPIWrapper
import logging
from langchain.agents.middleware import before_model, after_model
from langchain.agents.middleware import SummarizationMiddleware
#from langchain.middleware.summarization import SummarizationMiddleware
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
#from langchain_anthropic.middleware import anthropicPromptCachingMiddleware
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
import bs4
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


### Construct retriever ###
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [16]:
retriever.invoke("What is Task Decomposition?")

[Document(id='111a17f6-b312-429c-a6f9-cf29744baef6', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-f

In [3]:
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever =contextualize_q_prompt|llm|StrOutputParser()


In [17]:
# 4. Retrieval Step
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
def retrieve_docs(inputs):
    standalone_q = inputs["standalone_question"]
    docs = retriever.invoke(standalone_q)
    return {"docs": docs, "standalone_question": standalone_q, "input": inputs["input"], "chat_history": inputs["chat_history"]}

retrieval_chain = RunnableLambda(retrieve_docs)

# 5. QA Prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(x["docs"])
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# 6. Full Chain
rag_chain = (
    RunnablePassthrough.assign(
        standalone_question=history_aware_retriever
    )
    | retrieval_chain
    | qa_chain
)

In [28]:

from langchain.messages import HumanMessage,SystemMessage,AIMessage
chat_history = []

def ask_question(query):
    global chat_history

    response = rag_chain.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # ✅ Append AFTER response
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response))

    return response

In [29]:
ask_question("What is Task Decomposition?")


'Task decomposition is a technique used to break down complex tasks into smaller and simpler steps. This approach helps in managing and solving difficult problems by dividing them into more manageable subtasks. One common method for task decomposition is the Chain of Thought (CoT) technique, where models are prompted to think step by step to decompose hard tasks. Another extension of this technique is the Tree of Thoughts, which explores multiple reasoning possibilities at each step by creating a tree structure of multiple thoughts per step. Task decomposition can also be achieved through simple prompting, task-specific instructions, or with the help of human inputs. Additionally, there is a distinct approach called LLM+P, which involves using an external classical planner for long-horizon planning by translating the problem into PDDL, generating a PDDL plan, and then translating it back into natural language.'

In [30]:
ask_question("What are common ways of doing it?")


'Common ways of task decomposition include:\n1. Using the Chain of Thought (CoT) technique: Models are prompted to think step by step to decompose hard tasks into smaller and simpler steps.\n2. Utilizing the Tree of Thoughts approach: This extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure of multiple thoughts per step.\n3. Simple prompting with LLM: Using simple prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ" to guide the model in breaking down tasks.\n4. Task-specific instructions: Providing specific instructions tailored to the task at hand, such as "Write a story outline" for writing a novel.\n5. Human inputs: Involving human input to assist in breaking down complex tasks into more manageable subtasks.'

In [32]:
chat_history

[HumanMessage(content='What is Task Decomposition?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Task decomposition is a technique used to break down complex tasks into smaller and simpler steps. This approach helps in managing and solving difficult problems by dividing them into more manageable subtasks. One common method for task decomposition is the Chain of Thought (CoT) technique, where models are prompted to think step by step to decompose hard tasks. Another extension of this technique is the Tree of Thoughts, which explores multiple reasoning possibilities at each step by creating a tree structure of multiple thoughts per step. Task decomposition can also be achieved through simple prompting, task-specific instructions, or with the help of human inputs. Additionally, there is a distinct approach called LLM+P, which involves using an external classical planner for long-horizon planning by translating the problem into PDDL, generating a PDDL plan, and then tr

In [ ]:
chat_history

[HumanMessage(content='What is Task Decomposition?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Task decomposition is a technique used to break down complex tasks into smaller and simpler steps. This approach helps in managing and solving difficult problems by dividing them into more manageable subtasks. One common method for task decomposition is the Chain of Thought (CoT) technique, where models are prompted to think step by step to decompose hard tasks. Another extension of this technique is the Tree of Thoughts, which explores multiple reasoning possibilities at each step by creating a tree structure of multiple thoughts per step. Task decomposition can also be achieved through simple prompting, task-specific instructions, or with the help of human inputs. Additionally, there is a distinct approach called LLM+P, which involves using an external classical planner for long-horizon planning by translating the problem into PDDL, generating a PDDL plan, and then tr

: 